<img src="../img/GTK_Logo_Social_Icon.jpg" width=175 align="right" />

# Worksheet 8.1 Attacking AI - Answers

In this lab, we will learn how to use the Adversarial Robustness Toolkit (ART) to launch various attacks against models. The first attack you will launch will be to create adversarial examples from a model.  These examples could be used to defeat a model, or control the model's behavior.

The documentation for ART can be found here: https://github.com/Trusted-AI/adversarial-robustness-toolbox/tree/main

In [ ]:
import numpy as np
import pandas as pd
import joblib
from art.attacks.evasion import DecisionTreeAttack, HopSkipJump
from art.estimators.classification import SklearnClassifier, BlackBoxClassifier
from art.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')
DATA_HOME = '../data'

## Decision Tree Attack
In this example, we are going to use the ART to attack a decision tree. The goal is to create adversarial examples which could be used to control the output of the model.  

Due to the nature of decision trees, it is not necessary to use gradient descent to discover adversarial examples and instead, it can be accomplished by tree traversals. This attack is a whitebox attack in that you need to have access to the actual model. 

This methodology was described in a paper by Papernot et al. in https://arxiv.org/abs/1605.07277. You can see this code in action here: https://github.com/Trusted-AI/adversarial-robustness-toolbox/blob/main/notebooks/attack_decision_tree.ipynb.

First we're going to load the model from a pickle file. 

In [ ]:
# Load the classifier from the pickle file
with open(f"{DATA_HOME}/dga_decision_tree.pkl", "rb") as file:
    clf = joblib.load(file)

In [ ]:
clf

We will also need some training data.  In this case, we'll use the data that was used to train the original model, but this is not necessary. 

In [ ]:
df = pd.read_csv(f'{DATA_HOME}/dga_features_final_df.csv')
target = df['isDGA']
feature_matrix = df.drop(['isDGA'], axis=1)
feature_matrix_train, feature_matrix_test, target_train, target_test = train_test_split(
    feature_matrix, target, test_size=0.25, random_state=42)

### Step 1:  Create the ART Classifier
As a first step, we need to use ART to create an "adversarial" classifier.  Use the `SklearnClassifier` module from ART. 

In [ ]:
# Use ART's SklearnClassifier to wrap the loaded model (clf) so it can be attacked.
# Store the wrapped model in a variable called adversarial_classifier.
# Your code here...

### Step 2:  Attack!!!
Now that you've created an adversarial classifier the next step is to train that adversarial classifier.  Use the `DecisionTreeAttack` module in ART to launch an attack, then call the `generate()` with the `feature_matrix_train` and `target_train` datasets.  The `generate()` method can be called either with only the feature matrix alone or you can call it with a list of desired targets.  

For our example, let's say that we want all the results to be classified as legitimate, we're going to pass it a numpy array of 1500 `0` for a target vector.


Note: You will have to call the `.to_numpy()` methods on these datasets when you pass them to ART.


This step generates a lot of future warnings. For this exercise we have suppressed them, however scikit-learn will throw warnings when you mix numpy arrays and dataframes.  The way to avoid this is to actually train your models on numpy arrays.  To do that, during the training process, convert the dataframe to a numpy array with the `.to_numpy()` method.

In [ ]:
# The model's classes are ['dga', 'legit'], so 'legit' is class index 1.
# Target every sample as 'legit' so the malicious (DGA) domains evade detection.
all_legit = np.array([1] * len(feature_matrix_train))

# Create a DecisionTreeAttack from your adversarial_classifier, then call its
# generate() method on feature_matrix_train (as a numpy array), passing all_legit
# as the target labels. Store the output in adversarial_data.
# Your code here...

### Step 3:  Evaluate the Performance
At this point you should have a dataset of adversarial examples that produce exclusively legit classifications.  Now try running that through the original classifier and making a classification report to see how we did.

**Baseline first.** Before looking at the attack, classify the *original* (unperturbed) training data so we have something to compare against.

In [ ]:
# Baseline: the model on the clean training data
baseline_preds = clf.predict(feature_matrix_train.to_numpy())
print(classification_report(target_train, baseline_preds))

In [ ]:
# Run adversarial_data through the original classifier (clf) and print a
# classification_report comparing target_train to those predictions.
# Compare the result to the baseline above.
# Your code here...

In [ ]:
# Display a confusion matrix comparing target_train to your adversarial predictions.
# Your code here...

If you did this correctly, you should get predictions that are entirely the `legit` class.  This shows how you are able to generate adversarial data that can be crafted to direct the decisions of a model.

## BlackBox Adversarial Attack
Now that you've successfully launched a white box adversarial attack, let's try a blackbox attack. We're going to use the `HopSkipJump` attack from Jianbo et al. (2019). This is a powerful black-box attack that only requires final class prediction, and is an advanced version of the boundary attack.

Paper link: https://arxiv.org/abs/1904.02144

In order to execute this attack, we will need a `predict()` function which calls a trained model and returns the predictions. In our example, the `predict()` function is simply a wrapper for our trained classifier, however, this same technique could be used with a true blackbox model where only the predictions are accessible. In that case, the `predict()` function would contain API calls or something similar.

In [ ]:
def predict(x):
    '''
    Call the model and return the predictions.  This function could contain calls to a true 
    blackbox model, but in this example, is calling our pre-trained model.
    '''
    x = np.array(x)
    preds = clf.predict(x)
    # The model returns string labels ('dga'/'legit'); ART needs integer classes.
    preds = np.array([label_map[p] for p in preds])
    return to_categorical(preds, nb_classes=2)

### Step 1:  Create the BlackBox Classifier
In order to execute the attack we need to first create a `BlackBoxClassifier`.  At a minimum, we need to pass the predict function, the number of features and the number of possible classes.

In [ ]:
# Create a BlackBoxClassifier from the predict function defined above. It needs:
#   - the predict function
#   - the input shape: feature_matrix_train.iloc(0)[0].shape
#   - nb_classes=2
# Store it in blackbox_clf.
# Your code here...

### Step 2:  ATTACK!!  Generate Adversarial Examples
The next step is to create the `HopSkipJump` object to launch the attack.  This follows a similar pattern as the previous attack where you create the `attack` object, then call the `generate()` method passing the testing features (`feature_matrix_test`).  This will generate an array of adversarial examples.  

For our use case, let's say that we want to generate adversarial examples that skew towards one class. In the `HopSkipJump` object, set `targeted=True` which forces the attack to generate examples for one class only. 


NOTE: You will have to convert the testing features to a numpy array like this:
```python
feature_matrix_test.to_numpy()
```

In [ ]:
label_map = {
    "legit": 0,
    "dga": 1,
}

In [ ]:
final_features_test = feature_matrix_test.to_numpy().astype(np.float32)
final_target_test = target_test.map(label_map).to_numpy().astype(int)

# Create a HopSkipJump attack against blackbox_clf. Use an untargeted attack
# (targeted=False) so it pushes each sample across the decision boundary.
# Suggested parameters: norm=np.inf, max_iter=100, max_eval=100, init_eval=100, init_size=100.
# Then call generate() on final_features_test and store the result in
# adversarial_data_blackbox. (This step can take a minute or two to run.)
# Your code here...

### Step 3:  Evaluate the Attack
Now that you have a set of adversarial data, let's make some predictions with that data and see how effective it is in predicting the model output.  You won't be able to use Yellowbrick here because the `BlackBoxClassifier` does not implement the `fit()` method. 

For this final step, make the predictions, then create a confusion matrix of this data to evaluate your BlackBox model's performance.

**Baseline first.** Classify the *original* (unperturbed) test data so we can measure how much the attack degrades performance.

In [ ]:
# Baseline: the black-box model on the clean test data
baseline_preds_blackbox = np.argmax(blackbox_clf.predict(final_features_test), axis=1)
print(classification_report(final_target_test, baseline_preds_blackbox))

In [ ]:
# Predict on adversarial_data_blackbox with blackbox_clf and take argmax over axis=1
# to get integer class predictions. Store them in adversarial_predictions.
# Then compute the fraction of predictions that changed compared with
# baseline_preds_blackbox and print it.
# Your code here...

In [ ]:
# Display a confusion matrix comparing final_target_test to adversarial_predictions.
# Your code here...

How did the model do?  If the attack succeeded, the confusion matrix should be almost entirely *off* the diagonal: the black-box model now misclassifies nearly every adversarial example, even though it classified the clean data almost perfectly.  That collapse from the baseline is the measure of the attack's effectiveness.